In [1]:
import pandas as pd

In [2]:
import requests
from bs4 import BeautifulSoup

## Scraping an HTML table

Open the URL https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population and scroll down until you see a table of the cities in the U.S. with population over 100,000 (as of Jul 1, 2025). We'll use Beautiful Soup to scrape information from this table.

1\. Read in the HTML from the URL using the `requests` library.

In [5]:
url = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"
headers = {"User-Agent": "GSB5544-PracticeActivity/5.2"}
response = requests.get(url, headers=headers)
response.status_code

200

2\. Use Beautiful Soup to parse this string into a tree called `soup`

In [6]:
soup = BeautifulSoup(response.text, "html.parser")

3\. Determine how many tables are in `soup`. (Hint: use `find_all("table")`.)

In [7]:
len(soup.find_all("table"))

10

4\. There are several tables included in `soup`, so we need to narrow it down. Go to the cities table Wikipedia page and "Inspect" it. What are the attributes (class, style) of this table?



In [9]:
city_table = soup.find("table", class_="col1left")

print("class:", " ".join(city_table["class"]))
print("style:", city_table["style"])

class: sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center
style: text-align:right


5\. You should find that the cities table on the Wikipedia page corresponds to the element

```
<table class="sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center" style="text-align:right">
```

How many tables in `soup` have these attributes?

In [11]:
len(soup.find_all("table", attrs={"class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center","style": "text-align:right"}))

1

6\. There should only be 1 table of this type, so we just need to select it. The following code finds all tables with the desired attributes and then selects the first (only) one to store as `table`. (You just need to run this.)

In [12]:
table = soup.find_all("table",
                  attrs={
                      "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
                      "style": "text-align:right"}
                  )[0]

7\. Our goal is now to scrape the information in `table` to create a Pandas data frame with one row for each city and columns for:

- city
- state
- population (2025 estimate)
- 2020 land area (sq mi).

First, let's just see how to scrape the information for New York City. Starting from `table` create an object, called `city`, that contains the information just for New York City.

Hints: Inspect the source; what kind of tag represents each row? Find all tags of this type in `table` and select the first one that corresponds to a city. Note that the first 3 rows of the table are headers.

In [13]:
city = table.find_all("tr")[3]
city

<tr id="mw0g">
<td id="mw0w" style="background-color:#cfecec"><a href="https://en.wikipedia.org/wiki/New_York_City" id="mw1A" rel="mw:WikiLink" title="New York City">New York</a><sup about="#mwt57" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-5"},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"1":{"wt":"Since 1898, the [[New York City|City of New York]], New York, has comprised [[borough of New York City|five boroughs]] with [[consolidated city-county|consolidated borough–county governments]] (2025 population estimates):\n* [[Brooklyn|The Borough of Brooklyn]] and [[Brooklyn|Kings County]]\n: (pop. 2,653,963)\n* [[Queens|The Borough of Queens]] and [[Queens|Queens County]]\n: (pop. 2,358,182)\n* [[Manhattan|The Borough of Manhattan]] and [[Manhattan|New York County]]\n: (pop. 1,664,862)\n* [[The Bronx|The Borough of the Bronx]] and [[The Bronx|Bronx County]]\n: (pop. 1,406,332)\n

8\. Starting with `city` extract the city's name and store it as `name`.

Hints: Inspect the source; what tag represents the cells within a row? Find all tags of this type and extract the text corresponding to the cell with the city's name.

In [14]:
name = city.find_all("td")[0].find("a").text
name

'New York'

9\. Extract the city's state and store it as `state`.

In [15]:
state = city.find_all("td")[1].find("a").text
state

'NY'

10\. Extract the city's population and store is as `population`.

In [16]:
population = int(city.find_all("td")[2].text.replace(",", ""))
population

8584629

11\. Extract the city's area and store is as `area`.

In [17]:
area = float(city.find_all("td")[5].text.replace(",", ""))
area

300.5

12\. Now put the steps for a single city into a loop to extract the information for all cities in `table` and create a data frame.

Hints:
- Start with an empty list named `rows`
- Write a loop that starts `for city in ...` and replace `...` code that finds all the table rows. (Use what you did in part 7, but don't just select one row. Select all rows except for the 3 header rows.)
- Use your code from 8-11 to extract the information for the city
- And append it to `rows` as "name", "state", "population", "area"
- Convert `rows` into a Pandas data frame. You should obtain a data frame with 348 rows and 4 columns.


In [19]:
rows = []

for city in table.find_all("tr")[3:]:
    cells = city.find_all("td")
    name = cells[0].find("a").text
    state = cells[1].find("a").text
    population = int(cells[2].text.replace(",", ""))
    area = float(cells[5].text.replace(",", ""))
    rows.append({"name": name, "state": state, "population": population, "area": area})

df_cities = pd.DataFrame(rows)
df_cities

,name,state,population,area
0,New York,NY,8584629,300.5
1,Los Angeles,CA,3869089,469.5
2,Chicago,IL,2731585,227.7
3,Houston,TX,2397315,640.4
4,Phoenix,AZ,1665481,518.0
...,...,...,...,...
343,San Angelo,TX,100640,59.7
344,Edmond,OK,100479,84.6
345,Davenport,IA,100358,63.8
346,Deltona,FL,100267,37.3


13\. Use the Pandas command `pd.read_html` can be used to scrape the table from the webpage. Note: `read_html` will return all the tables, so you will need to narrow your request using attributes. You don't need to worry about selecting columns; just scrape the whole table.

In [22]:
%pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 15.3 MB/s  0:00:00 15.0 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [24]:
from io import StringIO

df_cities_html = pd.read_html(StringIO(response.text), attrs={"class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center", "style": "text-align:right"})[0]
df_cities_html = df_cities_html.dropna(how="all").reset_index(drop=True)
df_cities_html

Municipality  ST 2025 estimate 2020 census  Change 2020 land area          \
    Municipality  ST 2025 estimate 2020 census  Change            mi2     km2   
0    New York[c]  NY     8584629.0   8804190.0  −2.49%          300.5   778.3   
1    Los Angeles  CA     3869089.0   3898747.0  −0.76%          469.5  1216.0   
2        Chicago  IL     2731585.0   2746388.0  −0.54%          227.7   589.7   
3        Houston  TX     2397315.0   2304580.0  +4.02%          640.4  1658.6   
4        Phoenix  AZ     1665481.0   1608139.0  +3.57%          518.0  1341.6   
..           ...  ..           ...         ...     ...            ...     ...   
343   San Angelo  TX      100640.0     99893.0  +0.75%           59.7   154.6   
344       Edmond  OK      100479.0     94428.0  +6.41%           84.6   219.1   
345    Davenport  IA      100358.0    101724.0  −1.34%           63.8   165.2   
346      Deltona  FL      100267.0     93692.0  +7.02%           37.3    96.6   
347     Longmont  CO      100109.0     98885.0  +1.24%           28.8    74.6   

    2020 density                                        Location  
           / mi2    / km2                               Location  
0        29298.0  11312.0    40°40′N 73°56′W﻿ / ﻿40.66°N 73.94°W  
1         8304.0   3206.0  34°01′N 118°25′W﻿ / ﻿34.02°N 118.41°W  
2        12061.0   4657.0    41°50′N 87°41′W﻿ / ﻿41.84°N 87.68°W  
3         3599.0   1390.0    29°47′N 95°23′W﻿ / ﻿29.79°N 95.39°W  
4         3105.0   1199.0  33°34′N 112°05′W﻿ / ﻿33.57°N 112.09°W  
..           ...      ...                                    ...  
343       1673.0    646.0  31°26′N 100°27′W﻿ / ﻿31.44°N 100.45°W  
344       1116.0    431.0    35°40′N 97°25′W﻿ / ﻿35.67°N 97.41°W  
345       1594.0    615.0    41°34′N 90°36′W﻿ / ﻿41.56°N 90.60°W  
346       2512.0    970.0    28°55′N 81°13′W﻿ / ﻿28.91°N 81.21°W  
347       3434.0   1326.0  40°10′N 105°06′W﻿ / ﻿40.17°N 105.10°W  

[348 rows x 10 columns]

## Scraping from multiple webpages

We will scrape the hockey statistics from this website: https://www.scrapethissite.com/pages/forms/. Notice that the information is spread over many pages.

1\. Scrape the information from the first page with Beatiful Soup.

In [25]:
hockey_url = "https://www.scrapethissite.com/pages/forms/"
response = requests.get(hockey_url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")
response.status_code

200

2\. Find the main table on this page and store it as `table`.

In [26]:
table = soup.find("table")
table

<table class="table">
<tr>
<th>
                            Team Name
                        </th>
<th>
                            Year
                        </th>
<th>
                            Wins
                        </th>
<th>
                            Losses
                        </th>
<th>
                            OT Losses
                        </th>
<th>
                            Win %
                        </th>
<th>
                            Goals For (GF)
                        </th>
<th>
                            Goals Against (GA)
                        </th>
<th>
                            + / -
                        </th>
</tr>
<tr class="team">
<td class="name">
                            Boston Bruins
                        </td>
<td class="year">
                            1990
                        </td>
<td class="wins">
                            44
                        </td>
<td class="losses">
                            2

3\. Extract the information from the cells of this table into a Pandas data frame.

In [27]:
columns = [cell.text.strip() for cell in table.find_all("th")]
rows = []

for team in table.find_all("tr", class_="team"):
    cells = team.find_all("td")
    rows.append([cell.text.strip() for cell in cells])

df_hockey = pd.DataFrame(rows, columns=columns)

for column in columns[1:]:
    df_hockey[column] = pd.to_numeric(df_hockey[column], errors="coerce")

df_hockey

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,NaN,0.550,299,264,35
1,Buffalo Sabres,1990,31,30,NaN,0.388,292,278,14
2,Calgary Flames,1990,46,26,NaN,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,NaN,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,NaN,0.425,273,298,-25
5,Edmonton Oilers,1990,37,37,NaN,0.463,272,272,0
6,Hartford Whalers,1990,31,38,NaN,0.388,238,276,-38
7,Los Angeles Kings,1990,46,24,NaN,0.575,340,254,86
8,Minnesota North Stars,1990,27,39,NaN,0.338,256,266,-10
9,Montreal Canadiens,1990,39,30,NaN,0.487,273,249,24


4\. But this only represents the first page of data. There are many pages of data. How do we scrape all of the data?

We could switch to different pages by modifying the `page_num` parameter in the URL.

Alternatively, we can just grab the links at the bottom of the page.

In [30]:
pagination = soup.find("ul", attrs={"class": "pagination"})
links = pagination.find_all("a")

Let's take a look at the links found.

In [31]:
for link in links:
  print(link.attrs["href"])

/pages/forms/?page_num=1
/pages/forms/?page_num=2
/pages/forms/?page_num=3
/pages/forms/?page_num=4
/pages/forms/?page_num=5
/pages/forms/?page_num=6
/pages/forms/?page_num=7
/pages/forms/?page_num=8
/pages/forms/?page_num=9
/pages/forms/?page_num=10
/pages/forms/?page_num=11
/pages/forms/?page_num=12
/pages/forms/?page_num=13
/pages/forms/?page_num=14
/pages/forms/?page_num=15
/pages/forms/?page_num=16
/pages/forms/?page_num=17
/pages/forms/?page_num=18
/pages/forms/?page_num=19
/pages/forms/?page_num=20
/pages/forms/?page_num=21
/pages/forms/?page_num=22
/pages/forms/?page_num=23
/pages/forms/?page_num=24
/pages/forms/?page_num=1


Now we can loop over `links` to make a request to the url for each page and scrape the data into a table similar to what we did for the first page. Write such a loop to extract the data and create a Pandas data frame.


A few technicalities:

- You might need to skip the "previous" and "next" buttons
- So you don't keep repeating headers, you will want to skip rows that don't represent teams.

In [32]:
import time

page_urls = []
for link in links:
    page_url = "https://www.scrapethissite.com" + link["href"]
    if page_url not in page_urls:
        page_urls.append(page_url)

all_rows = []

for page_url in page_urls:
    page_response = requests.get(page_url, headers=headers, timeout=30)
    page_response.raise_for_status()
    page_soup = BeautifulSoup(page_response.text, "html.parser")

    for team in page_soup.find_all("tr", class_="team"):
        cells = team.find_all("td")
        all_rows.append([cell.text.strip() for cell in cells])

    print("Finished:", page_url)
    time.sleep(1)

df_hockey_all = pd.DataFrame(all_rows, columns=columns)

for column in columns[1:]:
    df_hockey_all[column] = pd.to_numeric(df_hockey_all[column], errors="coerce")

df_hockey_all

Finished: https://www.scrapethissite.com/pages/forms/?page_num=1
Finished: https://www.scrapethissite.com/pages/forms/?page_num=2
Finished: https://www.scrapethissite.com/pages/forms/?page_num=3
Finished: https://www.scrapethissite.com/pages/forms/?page_num=4
Finished: https://www.scrapethissite.com/pages/forms/?page_num=5
Finished: https://www.scrapethissite.com/pages/forms/?page_num=6
Finished: https://www.scrapethissite.com/pages/forms/?page_num=7
Finished: https://www.scrapethissite.com/pages/forms/?page_num=8
Finished: https://www.scrapethissite.com/pages/forms/?page_num=9
Finished: https://www.scrapethissite.com/pages/forms/?page_num=10
Finished: https://www.scrapethissite.com/pages/forms/?page_num=11
Finished: https://www.scrapethissite.com/pages/forms/?page_num=12
Finished: https://www.scrapethissite.com/pages/forms/?page_num=13
Finished: https://www.scrapethissite.com/pages/forms/?page_num=14
Finished: https://www.scrapethissite.com/pages/forms/?page_num=15
Finished: https://w

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,NaN,0.550,299,264,35
1,Buffalo Sabres,1990,31,30,NaN,0.388,292,278,14
2,Calgary Flames,1990,46,26,NaN,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,NaN,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,NaN,0.425,273,298,-25
...,...,...,...,...,...,...,...,...,...
577,Tampa Bay Lightning,2011,38,36,8.0,0.463,235,281,-46
578,Toronto Maple Leafs,2011,35,37,10.0,0.427,231,264,-33
579,Vancouver Canucks,2011,51,22,9.0,0.622,249,198,51
580,Washington Capitals,2011,42,32,8.0,0.512,222,230,-8
